<a href="https://colab.research.google.com/github/codemiitkumar/AAI-501-Group-3-EV-Charging-And-Revenue-Optimization/blob/main/EV_Charging_Patterns_and_Revenue_Optimisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Smart EV Charging: Demand Prediction, Dynamic Pricing, and Charging Schedule Optimization

## AAI-501 Final Project
### Group 3

In [4]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [5]:
warnings.filterwarnings("ignore")

##Import Data File

In [13]:
OUTPUT_DIR = Path("ev_project_output")
OUTPUT_DIR.mkdir(exist_ok=True)

In [14]:
RANDOM_STATE = 42

In [15]:
df = pd.read_csv("/content/ChargingRecords.csv")

In [16]:
df.head()

,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand
0,0,1,1,hotel,0,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54,2022-09-15 23:59,185,20.36
1,0,1,1,hotel,0,2022-09-14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01,2022-09-14 21:31,90,10.19
2,0,1,1,hotel,0,2022-09-14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54,2022-09-14 19:54,60,6.78
3,0,1,1,hotel,0,2022-09-29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32,2022-09-30 0:16,344,37.65
4,0,1,1,hotel,0,2022-09-25,19:30:15,2022-09-26,0:30:14,2022-09-25 19:30,2022-09-26 0:30,300,33.81


In [18]:
print(f"Rows: {len(df):,}")

Rows: 72,856


In [19]:
print(f"Columns: {len(df.columns)}")

Columns: 13


In [21]:
print("Columns:")
print(df.columns.tolist())

Columns:
['UserID', 'ChargerID', 'ChargerCompany', 'Location', 'ChargerType', 'StartDay', 'StartTime', 'EndDay', 'EndTime', 'StartDatetime', 'EndDatetime', 'Duration', 'Demand']


## 2. DATA CLEANING

In [23]:
df.columns = [c.strip() for c in df.columns]

In [25]:
df["StartDatetime"] = pd.to_datetime(
    df["StartDatetime"], errors="coerce"
)

In [26]:
df["EndDatetime"] = pd.to_datetime(
    df["EndDatetime"], errors="coerce"
)

Numeric conversion.

In [27]:
numeric_columns = [
    "UserID",
    "ChargerID",
    "ChargerCompany",
    "ChargerType",
    "Duration",
    "Demand",
]

In [28]:
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

Remove exact duplicates.

In [30]:
before = len(df)

In [31]:
df = df.drop_duplicates().copy()

In [32]:
print(f"Exact duplicates removed: {before - len(df):,}")

Exact duplicates removed: 0


Remove rows with essential missing values.

In [33]:
essential = [
    "UserID",
    "ChargerID",
    "Location",
    "ChargerType",
    "StartDatetime",
    "EndDatetime",
    "Duration",
    "Demand",
]

In [34]:
before = len(df)

In [35]:
df = df.dropna(subset=essential).copy()

In [36]:
print(f"Rows removed for missing essential values: {before - len(df):,}")

Rows removed for missing essential values: 0


In [37]:
len(df)

72843

Recalculate actual duration from timestamps.

In [38]:
df["CalculatedDuration"] = (
    df["EndDatetime"] - df["StartDatetime"]
).dt.total_seconds() / 60

Identify timestamp inconsistencies.

In [39]:
df["DurationDifference"] = (
    df["Duration"] - df["CalculatedDuration"]
).abs()

Valid physical sessions

In [40]:
valid_mask = (
    (df["EndDatetime"] > df["StartDatetime"])
    & (df["CalculatedDuration"] > 0)
    & (df["Demand"] > 0)
)

In [42]:
invalid_count = (~valid_mask).sum()

In [44]:
print(f"Invalid timestamp/demand rows removed: {invalid_count:,}")

Invalid timestamp/demand rows removed: 547


In [45]:
df = df.loc[valid_mask].copy()

Use timestamp-derived duration because it is internally consistent.

In [46]:
df["Duration"] = df["CalculatedDuration"]

Extreme-value checks.

In [47]:
print("Cleaned descriptive statistics:")
print(
    df[["Duration", "Demand"]]
    .describe()
    .round(2)
)

Cleaned descriptive statistics:
       Duration    Demand
count  72296.00  72296.00
mean     156.68     17.56
std      159.20     13.46
min        1.00      0.01
25%       41.00      7.68
50%      115.00     14.25
75%      205.00     23.20
max     4717.00     97.00


In [49]:
print("Final useable Data shape:", df.shape)

Final useable Data shape: (72296, 15)


In [50]:
df.to_csv(OUTPUT_DIR / "cleaned_charging_records.csv", index=False)